Only For bns

In [2]:
import sys
import os

print("Python:", sys.executable)
print("Working directory:", os.getcwd())

!nvidia-smi

Python: /usr/bin/python3
Working directory: /content
/bin/bash: line 1: nvidia-smi: command not found


In [3]:
%pip install PyMuPDF pypdf huggingface_hub pymongo python-dotenv sentence-transformers tqdm ipykernel langchain-text-splitters langchain langgraph langchain-community langchain-groq fastapi[all] uvicorn[standard]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 MB 37.4 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 38.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 69.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.6/807.6 kB 27.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 58.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 928.0/928.0 kB 30.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 63.9 MB/s  0:00:00m0:00:0100:01
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25/25 [PyMuPDF]4/25 [PyMuPDF]n-community]]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviou

In [4]:
import fitz
import re
import numpy as np

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [11]:
PDF_PATH = r"./statutes/New/bns/THE_BHARATIYA_NYAYA_SANHITA_2023.pdf"

In [12]:
def extract_main_text(page):

    blocks = page.get_text("blocks")

    kept_blocks = []

    for block in blocks:

        x0, y0, x1, y1, text, *_ = block

        # Main central statute column.
        # Removes marginal section titles.
        if x0 >= 110 and x1 <= 485 and 80 <= y0 <= 770:

            text = text.strip()

            if not text:
                continue

            # Remove Gazette header if captured
            if "THE GAZETTE OF INDIA EXTRAORDINARY" in text:
                continue

            if text.startswith("[Part II") or text.startswith("[PART II"):
                continue

            kept_blocks.append(
                (y0, x0, text)
            )

    # Correct reading order
    kept_blocks.sort(
        key=lambda x: (x[0], x[1])
    )

    return "\n".join(
        text
        for _, _, text in kept_blocks
    )

In [13]:
pdf = fitz.open(PDF_PATH)

pages = []

for page_number, page in enumerate(pdf, start=1):

    text = extract_main_text(page)

    pages.append({
        "page": page_number,
        "text": text
    })

print("Pages:", len(pages))

Pages: 102


In [14]:
full_text = ""

for page in pages:

    full_text += (
        f"\n<<<PAGE:{page['page']}>>>\n"
        + page["text"]
        + "\n"
    )

In [15]:
SECTION_PATTERN = r"(?m)^(\d{1,3})\.\s*"

In [16]:
matches = list(
    re.finditer(
        SECTION_PATTERN,
        full_text
    )
)

print("Section starts found:", len(matches))

Section starts found: 358


In [17]:
sections = []

for i, match in enumerate(matches):

    section_number = int(
        match.group(1)
    )

    start = match.start()

    if i + 1 < len(matches):
        end = matches[i + 1].start()
    else:
        end = len(full_text)

    raw_section = full_text[start:end]

    # Find all page markers inside this section
    page_numbers = [
        int(p)
        for p in re.findall(
            r"<<<PAGE:(\d+)>>>",
            raw_section
        )
    ]

    # Remove page markers from actual text
    section_text = re.sub(
        r"<<<PAGE:\d+>>>",
        "",
        raw_section
    ).strip()

    sections.append(
        Document(
            page_content=section_text,
            metadata={
                "law": "BNS",
                "section": str(section_number),
                "source": PDF_PATH,
                "pages": page_numbers,
                "document_type": "statute"
            }
        )
    )

In [18]:
numbers = [
    int(doc.metadata["section"])
    for doc in sections
]

print("Total sections:", len(numbers))
print("First:", numbers[:10])
print("Last:", numbers[-10:])

Total sections: 358
First: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Last: [349, 350, 351, 352, 353, 354, 355, 356, 357, 358]


In [19]:
expected = list(range(1, 359))

print(
    "All sections correct:",
    numbers == expected
)

All sections correct: True


In [20]:
section_21 = next(
    doc
    for doc in sections
    if doc.metadata["section"] == "21"
)

print(section_21.page_content)

21. Nothing is an offence which is done by a child above seven years of age and under
twelve years of age, who has not attained sufficient maturity of understanding to judge of
the nature and consequences of his conduct on that occasion.


In [21]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(model.max_seq_length)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2145.95it/s]


256


In [22]:
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(

    tokenizer=model.tokenizer,

    chunk_size=200,

    chunk_overlap=30,

    separators=[
        "\n\n",
        "\n(",
        "\nExplanation",
        "\nIllustration",
        "\nProvided",
        ". ",
        " "
    ]
)

In [23]:
final_chunks = []

for section in sections:

    section_text = section.page_content

    token_count = len(
        model.tokenizer.encode(
            section_text,
            add_special_tokens=False
        )
    )

    # Small section -> keep whole
    if token_count <= 200:

        final_chunks.append(
            Document(
                page_content=section_text,
                metadata={
                    **section.metadata,
                    "chunk_index": 0,
                    "parent_section": section.metadata["section"]
                }
            )
        )

    # Large section -> split only within that section
    else:

        child_texts = splitter.split_text(
            section_text
        )

        for i, text in enumerate(child_texts):

            final_chunks.append(
                Document(
                    page_content=text,
                    metadata={
                        **section.metadata,
                        "chunk_index": i,
                        "parent_section":
                            section.metadata["section"]
                    }
                )
            )

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (355 > 256). Running this sequence through the model will result in indexing errors


In [24]:
print("Sections:", len(sections))
print("Embedding chunks:", len(final_chunks))

Sections: 358
Embedding chunks: 637


In [25]:
for chunk in final_chunks:

    if chunk.metadata["section"] == "2":

        print(
            "Chunk:",
            chunk.metadata["chunk_index"]
        )

        print(
            chunk.page_content[:500]
        )

        print("=" * 70)

Chunk: 0
2. In this Sanhita, unless the context otherwise requires,––
(1) “act” denotes as well a series of acts as a single act;
(2) “animal” means any living creature, other than a human being;
(3) “child” means any person below the age of eighteen years;
Chunk: 1
(3) “child” means any person below the age of eighteen years;
(4) “counterfeit”.––A person is said to “counterfeit” who causes one thing to
resemble another thing, intending by means of that resemblance to practise deception,
or knowing it to be likely that deception will thereby be practised.
Explanation 1.—It is not essential to counterfeiting that the imitation should be
exact.
Explanation 2.—When a person causes one thing to resemble another thing,
and the resemblance is such that a person m
Chunk: 2
(5) “Court” means a Judge who is empowered by law to act judicially alone, or a
body of Judges which is empowered by law to act judicially as a body, when such
Judge or body of Judges is acting judicially;
(6) “death” means

In [26]:
embedding_texts = []

for chunk in final_chunks:

    text = f"""
Bharatiya Nyaya Sanhita, 2023.
BNS Section {chunk.metadata['section']}.

{chunk.page_content}
""".strip()

    embedding_texts.append(text)

In [27]:
embeddings = model.encode(
    embedding_texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 20/20 [00:23<00:00,  1.20s/it]


In [28]:
vectors = []

for i, embedding in enumerate(embeddings):

    chunk = final_chunks[i]

    vectors.append({

        "id":
            f"bns-{chunk.metadata['section']}-{chunk.metadata['chunk_index']}",

        "values":
            embedding.tolist(),

        "metadata": {

            "law": "BNS",

            "section":
                chunk.metadata["section"],

            "chunk_index":
                chunk.metadata["chunk_index"],

            "text":
                chunk.page_content,

            "pages":
                chunk.metadata["pages"],

            "source":
                chunk.metadata["source"]
        }
    })

In [30]:
from pymongo import MongoClient
from dotenv import load_dotenv
import os

load_dotenv()
client = MongoClient(os.getenv("MONGODB_URI"))

db = client["test_db"]
collection = db["test_collection"]

collection.insert_many(vectors)

print(
    "Documents:",
    collection.count_documents({})
)

Documents: 637


In [34]:
def local_search(query, k=5):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    scores = embeddings @ query_embedding

    top_indices = np.argsort(
        scores
    )[::-1][:k]

    results = []

    for index in top_indices:

        results.append({
            "score": float(scores[index]),
            "section":
                final_chunks[index].metadata["section"],
            "chunk_index":
                final_chunks[index].metadata["chunk_index"],
            "text":
                final_chunks[index].page_content
        })

    return results

In [35]:
from langchain_groq import ChatGroq
load_dotenv()

def test_answer(question, k=5):

    # 1. Retrieve relevant chunks
    results = local_search(question, k=k)

    # 2. Build context
    context = "\n\n".join(
        [
            f"""
BNS Section {r['section']}
Score: {r['score']:.4f}

{r['text']}
""".strip()
            for r in results
        ]
    )

    # 3. Build prompt
    prompt = f"""
You are a legal information assistant.

Answer ONLY from the provided BNS context.

Rules:
1. Mention the relevant BNS section number.
2. Do not invent any section or legal rule.
3. If the answer is not supported by the context, say:
   "Insufficient information in retrieved context."

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""
    llm = ChatGroq(groq_api_key= os.getenv("GROQ_API_KEY") , model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)
    # 4. Generate answer
    response = llm.invoke(prompt)

    # 5. Show retrieved sections
    print("QUESTION:")
    print(question)

    print("\nRETRIEVED SECTIONS:")
    for r in results:
        print(
            f"Section {r['section']} | Score: {r['score']:.4f}"
        )

    print("\nANSWER:")
    print(response.content)

    return {
        "question": question,
        "retrieved": results,
        "answer": response.content
    }

In [46]:
print(test_answer("What is organised crime under BNS Section 111?"))

QUESTION:
What is organised crime under BNS Section 111?

RETRIEVED SECTIONS:
Section 111 | Score: 0.6276
Section 190 | Score: 0.5538
Section 111 | Score: 0.5354
Section 3 | Score: 0.5179
Section 111 | Score: 0.5156

ANSWER:
**BNS Section 111** defines organised crime as follows:

- **Section 111(1)** – Any **continuing unlawful activity** (such as kidnapping, robbery, vehicle theft, extortion, land grabbing, contract killing, economic offence, cyber‑crimes, trafficking of persons, drugs, weapons or illicit goods or services, human trafficking for prostitution or ransom), carried out by any person or a group of persons acting in concert—whether singly or jointly, as a member of an **organised crime syndicate** or on behalf of such a syndicate—**by use of violence, threat of violence, intimidation, coercion, or any other unlawful means to obtain direct or indirect material benefit (including financial benefit)**, shall constitute **organised crime**.

- **Explanation – “organised crime 